After executing the cell above, a new file named 'Sample file.txt' will appear in your [drive.google.com](https://drive.google.com/) file list.

In [ ]:
!pip install transformer_lens


In [ ]:
import torch
from transformer_lens import HookedTransformer

device="cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"model loaded with {model.cfg.n_layers}layers and {model.cfg.n_heads} heads per layer.")

In [ ]:
dataset = [
    {"prompt": "3 + 5 =", "target": " 8", "corrupt": " 9", "difficulty": "Easy (1-step)"},
    {"prompt": "14 + 27 =", "target": " 41", "corrupt": " 31", "difficulty": "Medium (2-step w/ carry)"},
    {"prompt": "158 + 274 =", "target": " 432", "corrupt": " 422", "difficulty": "Hard (3-step w/ multi-carry)"}
]

In [ ]:
def evaluate_prompts(model, data):
    results = []
    for item in data:
        logits, cache = model.run_with_cache(item["prompt"])
        last_logit = logits[0, -1, :]
        pred_token_id = torch.argmax(last_logit).item()
        pred_str = model.to_string(pred_token_id)
        target_id = model.to_single_token(item["target"])
        corrupt_id = model.to_single_token(item["corrupt"])
        logit_diff = (last_logit[target_id] - last_logit[corrupt_id]).item()
        is_correct = (pred_str.strip() == item["target"].strip())
        results.append({
            "prompt": item["prompt"],
            "difficulty": item["difficulty"],
            "logit_diff": logit_diff,
            "correct": is_correct
        })
    return results

results = evaluate_prompts(model, dataset)
for r in results:
    print(r)


In [ ]:

few_shot_prefix = "1 + 1 = 2\n2 + 2 = 4\n5 + 3 = 8\n"

dataset = [
    {"prompt": few_shot_prefix + "3 + 5 =", "target": " 8", "corrupt": " 9", "difficulty": "Easy (1-step)"},
    {"prompt": few_shot_prefix + "14 + 27 =", "target": " 41", "corrupt": " 31", "difficulty": "Medium (2-step w/ carry)"},
    {"prompt": few_shot_prefix + "158 + 274 =", "target": " 432", "corrupt": " 422", "difficulty": "Hard (3-step w/ multi-carry)"}
]

results = evaluate_prompts(model, dataset)
for r in results:
    print(r)

In [ ]:
for item in dataset:
    logits, cache = model.run_with_cache(item["prompt"])
    pred_id = torch.argmax(logits[0, -1, :]).item()
    print(f"Prompt: {item['prompt']!r}")
    print(f"  Predicted token: {model.to_string(pred_id)!r}")



In [ ]:
few_shot_prefix = (
    "1 + 1 = 2\n"
    "2 + 2 = 4\n"
    "5 + 3 = 8\n"
    "6 + 7 = 13\n"
    "18 + 24 = 42\n"
)

dataset = [
    {"prompt": few_shot_prefix + "3 + 5 =", "target": " 8", "corrupt": " 9", "difficulty": "Easy (1-step)"},
    {"prompt": few_shot_prefix + "14 + 27 =", "target": " 41", "corrupt": " 31", "difficulty": "Medium (2-step w/ carry)"},
    {"prompt": few_shot_prefix + "158 + 274 =", "target": " 432", "corrupt": " 422", "difficulty": "Hard (3-step w/ multi-carry)"}
]

results = evaluate_prompts(model, dataset)
for r in results:
    print(r)


In [ ]:
import torch
import torch.nn.functional as F
from transformer_lens import HookedTransformer

# 1. Load Model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)

# 2. Define clean and corrupted prompts/tokens
# Clean prompt where the model successfully favors the correct answer
clean_prompt = "The capital of France is Paris. The capital of Germany is"
clean_target_token = model.to_single_token(" Berlin")
corrupt_target_token = model.to_single_token(" London")

# Corrupted prompt (e.g., swapped context or distractor)
corrupt_prompt = "The capital of Spain is Madrid. The capital of Germany is"

# Tokenize inputs
clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

# 3. Define Metric: Logit Difference
def compute_logit_diff(logits, correct_id=clean_target_token, incorrect_id=corrupt_target_token):
    """
    Computes the difference in logits between the correct and incorrect tokens
    at the final sequence position.
    """
    final_logits = logits[0, -1, :]
    return final_logits[correct_id] - final_logits[incorrect_id]

# 4. Run Clean and Corrupted Forwards to establish baseline
with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    corrupt_logits, corrupt_cache = model.run_with_cache(corrupt_tokens)

clean_metric = compute_logit_diff(clean_logits)
corrupt_metric = compute_logit_diff(corrupt_logits)
print(f"Clean Logit Diff: {clean_metric.item():.4f}")
print(f"Corrupt Logit Diff: {corrupt_metric.item():.4f}")

# 5. Define Activation Patching Hook Function
def patch_residual_stream(corrupted_activations, hook, position, layer, clean_cache):
    """
    Replaces a specific layer and sequence position's activation
    in the corrupted run with the corresponding activation from the clean run.
    """
    # hook.name format: "blocks.{layer}.hook_resid_post"
    corrupted_activations[:, position, :] = clean_cache[hook.name][:, position, :]
    return corrupted_activations

# Example: Patching layer 6, last token position
layer_to_patch = 6
target_position = -1
hook_name = f"blocks.{layer_to_patch}.hook_resid_post"

# Create a hook pointing to our patch function
from functools import partial
patch_hook = partial(patch_residual_stream, position=target_position, layer=layer_to_patch, clean_cache=clean_cache)

# Run model with the activation patch applied
patched_logits = model.run_with_hooks(
    corrupt_tokens,
    fwd_hooks=[(hook_name, patch_hook)]
)

patched_metric = compute_logit_diff(patched_logits)
print(f"Patched Logit Diff (Layer {layer_to_patch}): {patched_metric.item():.4f}")

In [ ]:
import torch
from transformer_lens import HookedTransformer
import matplotlib.pyplot as plt

# 1. Setup Model and Prompts
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)

clean_prompt = "The capital of France is Paris. The capital of Germany is"
corrupt_prompt = "The capital of Spain is Madrid. The capital of Germany is"

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

clean_target_id = model.to_single_token(" Berlin")
corrupt_target_id = model.to_single_token(" London")

def compute_logit_diff(logits):
    return (logits[0, -1, clean_target_id] - logits[0, -1, corrupt_target_id]).item()

# 2. Run clean and corrupt baselines
with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    corrupt_logits, _ = model.run_with_cache(corrupt_tokens)

clean_baseline = compute_logit_diff(clean_logits)
corrupt_baseline = compute_logit_diff(corrupt_logits)

# 3. Loop across all layers for residual stream patching
patched_logit_diffs = []
num_layers = model.cfg.n_layers

for layer in range(num_layers):
    hook_name = f"blocks.{layer}.hook_resid_post"

    def patch_hook(activations, hook):
        activations[:, -1, :] = clean_cache[hook.name][:, -1, :]
        return activations

    patched_logits = model.run_with_hooks(
        corrupt_tokens,
        fwd_hooks=[(hook_name, patch_hook)]
    )

    diff = compute_logit_diff(patched_logits)
    patched_logit_diffs.append(diff)

# 4. Plot and Save Results
plt.figure(figsize=(10, 5))
plt.plot(range(num_layers), patched_logit_diffs, marker='o', linestyle='-', color='b', label='Patched Layer')
plt.axhline(y=clean_baseline, color='g', linestyle='--', label='Clean Baseline')
plt.axhline(y=corrupt_baseline, color='r', linestyle='--', label='Corrupt Baseline')
plt.title("Layer-wise Activation Patching (Residual Stream)")
plt.xlabel("Layer Index")
plt.ylabel("Logit Difference")
plt.legend()
plt.grid(True)

# Save plot as a PNG image in Colab file explorer
plt.savefig("layer_patching_results.png", dpi=300, bbox_inches="tight")
plt.show()